In [83]:
import pandas as pd
import numpy as np
from numpy.random import Generator, PCG64
from joblib import Parallel, delayed
import altair as alt

rng = Generator(PCG64())

In [86]:
# all_df = pd.read_csv(
#     "https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/final-data/all_valid_responses_df.csv",
#     low_memory=False
# )
# all_df['item_id'] = all_df['question'] + all_df['image_file']
all_df = pd.read_csv("./dataframes/all_evaluated_responses_df.csv")
all_df = all_df.dropna(subset=['agent_response'])
all_df['item_id'] = all_df['question'] + all_df['image_file']

In [112]:
all_df[
    all_df['test_type'] == 'chartqa-test-continuous-human'
][['image_file','question']].drop_duplicates()['question'].nunique()

125

### Find Human-Model Correlation

In [103]:
def bootstrap_correlation(
    model_responses, 
    human_responses,
    n_iterations=1000,
):
    """Added bootstrap coorelation """

    def bootstrap_iteration(
        model_responses,
        human_responses,
        chosen_items
    ):
        model_filter_df = pd.concat(
            [model_responses[model_responses['item_id'] == item] for item in chosen_items]
        )

        huamn_filter_df = pd.concat(
            [human_responses[human_responses['item_id'] == item] for item in chosen_items]
        )

        item_level_model_responses = model_filter_df.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        item_level_human_responses = huamn_filter_df.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        if item_level_model_responses['is_correct'].sum() == 0:
            human_model_correlation = np.nan
        else:
            human_model_correlation = np.corrcoef(
                item_level_model_responses['is_correct'],
                item_level_human_responses['is_correct']
            )[1][0]
            
        return human_model_correlation

    items = model_responses['item_id'].unique()
    bootstrapped_correlations = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            model_responses,
            human_responses,
            rng.choice(items, len(items), replace=True)
        ) for _ in range(n_iterations)
    )

    # get rid of all NA values
    bootstrapped_correlations = [bc for bc in bootstrapped_correlations if not pd.isna(bc)]

    if len(bootstrapped_correlations) == 0:
        return [np.nan, np.nan]
        
    lower = np.percentile(bootstrapped_correlations, 2.5)
    upper = np.percentile(bootstrapped_correlations, 97.5)

    return [lower, upper]

def human_model_correlation(all_df):
    """
    Find correlation on model and human responses on 
    """
    
    # grouped_df = all_df.groupby(['item_id', 'agent_type', 'test_type'])['is_correct'].mean().reset_index()
    human_model_corr_df = []

    for test in all_df['test_type'].unique():
        df = all_df[all_df['test_type'] == test].copy()
        print(test)
        
        human_responses = df[df['agent_type'] == 'Human']
        for agent in all_df['agent_type'].unique():
            if agent == 'Human':
                continue
            else:
                model_responses = df[df['agent_type'] == agent].copy()
                human_responses_valid = human_responses[
                    human_responses['item_id'].isin(model_responses['item_id'].unique())
                ]
            
            item_level_model_responses = model_responses.groupby(
                ['item_id', 'test_type']
            )['is_correct'].mean().reset_index().sort_values(by='item_id')

            item_level_human_responses = human_responses_valid.groupby(
                ['item_id', 'test_type']
            )['is_correct'].mean().reset_index().sort_values(by='item_id')

            # if the model only has incorrect responses for all questions
            if item_level_model_responses['is_correct'].sum() == 0:
                human_model_correlation = np.nan
            else:
                human_model_correlation = np.corrcoef(
                    item_level_model_responses['is_correct'],
                    item_level_human_responses['is_correct']
                )[1][0]

            lower, upper = bootstrap_correlation(
                model_responses,
                human_responses_valid
            )

            human_model_corr_df.append({
                'model': agent,
                'test': test,
                'lower_correlation': lower,
                'upper_correlation': upper,
                'correlation': human_model_correlation
            })

    return pd.DataFrame(human_model_corr_df)

try:
    print("reading human model correlation")
    human_model_correlation_df = pd.read_csv("./dataframes/human_model_correlation_df.csv")
    # raise Error
except:
    print("general human model correlation")
    human_model_correlation_df = human_model_correlation(all_df)
    human_model_correlation_df.to_csv("./dataframes/human_model_correlation_df.csv")

reading human model correlation
general human model correlation
ggr
vlat
calvi
holf
holf2
chartqa-test-continuous-human


In [56]:
# human_model_correlation_df = human_model_correlation(all_df)

In [57]:
# human_model_correlation_df

### Find split-half reliability

In [102]:
def bootstrap_pairwise_correlation(
    human_responses,
    n_iterations=1000,
):
    """Split-half pairwise correlation bootstrapped over human responses"""

    def bootstrap_iteration(
        human_responses,
        chosen_items
    ):
        huamn_filter_df = pd.concat(
            [human_responses[human_responses['item_id'] == item] for item in chosen_items]
        ).sample(frac=1)

        human_half1 = huamn_filter_df.iloc[0:len(huamn_filter_df)//2]
        human_half2 = huamn_filter_df.iloc[len(huamn_filter_df)//2:]

        human_half1 = human_half1[human_half1['item_id'].isin(human_half2['item_id'])]
        human_half2 = human_half2[human_half2['item_id'].isin(human_half1['item_id'])]

        item_level_human_half1_responses = human_half1.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        item_level_human_half2_responses = human_half2.groupby(
            ['item_id', 'test_type']
        )['is_correct'].mean().reset_index().sort_values(by='item_id')

        human_model_correlation = np.corrcoef(
            item_level_human_half1_responses['is_correct'],
            item_level_human_half2_responses['is_correct']
        )[1][0]
            
        return human_model_correlation

    items = human_responses['item_id'].unique()
    bootstrapped_correlations = Parallel(n_jobs=-1)(
        delayed(bootstrap_iteration)(
            human_responses,
            rng.choice(items, len(items), replace=True)
        ) for _ in range(n_iterations)
    )

    bootstrapped_correlations = [bc for bc in bootstrapped_correlations if not pd.isna(bc)]

    if len(bootstrapped_correlations) == 0:
        return [np.nan, np.nan]
        
    lower = np.percentile(bootstrapped_correlations, 2.5)
    upper = np.percentile(bootstrapped_correlations, 97.5)
    mean = np.mean(bootstrapped_correlations)

    return [mean, lower, upper]

def human_human_correlation(all_df):
    """
    Find correlation on model and human responses on 

    """
    
    human_model_corr_df = []

    for test in all_df['test_type'].unique():
        df = all_df[all_df['test_type'] == test].copy()
        
        human_responses = df[df['agent_type'] == 'Human']
        
        mean, lower, upper = bootstrap_pairwise_correlation(
            human_responses
        )
        human_model_corr_df.append({
            'model': 'Human',
            'test': test,
            'lower_correlation': lower,
            'upper_correlation': upper,
            'correlation': mean
        })

    return pd.DataFrame(human_model_corr_df)

try:
    print("reading human model correlation")
    human_human_correlation_df = pd.read_csv("./dataframes/human_human_correlation_df.csv")
except:
    print("general human model correlation")
    human_human_correlation_df = human_human_correlation(all_df)
    human_human_correlation_df.to_csv("./dataframes/human_human_correlation_df.csv")

reading human model correlation
general human model correlation


In [93]:
# human_human_correlation_df

### Correlation Error Bars

In [115]:
def create_human_machine_correlation_figure(correlation_df, tests, domain_order, domain_color=[]):
    """ Generate chart for point estimate and confidence intervals overlayed with item level scatteplots
    """
    charts = []
    for test in tests:
        df = correlation_df[correlation_df['test'] == test].copy()

        if test != 'ggr':
            axis =  alt.Axis(labels=False)
        else:
            axis = alt.Axis()
        
        mean_point_plot = alt.Chart(df).mark_point(filled=True, size=75, opacity=1).encode(
            x=alt.X('model:N', title='Category', scale=alt.Scale(domain=domain_order)),
            y=alt.Y('correlation:Q', title='', scale=alt.Scale(domain=[-1, 1]), axis=axis),
            color=alt.Color('model:N', scale=alt.Scale(domain=domain_order, range=domain_color), legend=None),
        )
        
        error_bars = alt.Chart(df).mark_rule(strokeWidth=2).encode(
            x=alt.X('model:N'),
            y=alt.Y('lower_correlation:Q'),
            y2='upper_correlation:Q',
            color=alt.Color('model:N', scale=alt.Scale(domain=domain_order, range=domain_color), legend=None)
        )

        split_half_band = alt.Chart(df[df['model'] == 'Human']).mark_rect(color=model_configs[-1]['color'], opacity=0.2).encode(
            y=alt.Y(f'lower_correlation:Q'),
            y2=(f'upper_correlation:Q'),
        )
    
        split_half_mean = alt.Chart(df[df['model'] == 'Human']).mark_rule(color=model_configs[-1]['color'], strokeDash=[5, 5]).encode(
            y=alt.Y(f'correlation:Q'),
            strokeWidth=alt.value(2)
        )

        if len(test) > 7:
            test = test[:7]
        
        chart = (mean_point_plot + error_bars + split_half_band + split_half_mean).properties(height=200, width=150, title=test)

        charts.append(chart)


    return alt.hconcat(*charts) #.resolve_scale(color='independent')

tests = ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']
domain_order = [mc['model'] for mc in model_configs if mc['model'] != 'Human']
domain_color = [mc['color'] for mc in model_configs if mc['model'] != 'Human']

correlation_df = pd.concat([
    human_model_correlation_df,
    human_human_correlation_df
])

human_machine_correlation_figure = create_human_machine_correlation_figure(
    correlation_df,
    tests=tests,
    domain_order=domain_order, 
    domain_color=domain_color,
)
human_machine_correlation_figure.save("./figures/human_machine_correlation.pdf")
human_machine_correlation_figure

alt.HConcatChart(...)

In [99]:
human_human_correlation_df

,model,test,lower_correlation,upper_correlation,correlation
0,Salesforce/blip2-flan-t5-xl,ggr,0.027889,0.304459,0.139554
1,Salesforce/blip2-flan-t5-xxl,ggr,0.019399,0.295929,0.139554
2,llava-hf/llava-1.5-7b-hf,ggr,NaN,NaN,NaN
3,llava-hf/llava-1.5-13b-hf,ggr,-0.001081,0.318035,0.139554
4,liuhaotian/llava-v1.6-34b,ggr,-0.810496,0.238019,0.038864
5,google/pix2struct-chartqa-base,ggr,NaN,NaN,NaN
6,google/matcha-chartqa,ggr,NaN,NaN,NaN
7,GPT-4V,ggr,-0.058719,0.383023,0.190099
8,Salesforce/blip2-flan-t5-xl,vlat,0.018994,0.436627,0.228250
9,Salesforce/blip2-flan-t5-xxl,vlat,-0.080282,0.273785,0.098217
